<a href="https://colab.research.google.com/github/ibrahim0015/Internship/blob/main/Neural%20Networks/CIFAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import torch

import torchvision
import torchvision.transforms as transforms

# Transform: convert images to tensors and normalize

# Training transform (with augmentation)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Test transform (no augmentation)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


# CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform
)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)


In [20]:
device = torch.device('cuda' if torch.cuda.is_available else 'cpu')
print(f'using device: {device}')

X_train = torch.tensor(trainset.data,dtype=torch.float32)
y_train =torch.tensor(trainset.targets,dtype=torch.long)
X_test = torch.tensor(testset.data,dtype=torch.float32)
y_test = torch.tensor(testset.targets,dtype=torch.long)



using device: cuda


In [21]:
from torch.utils.data import DataLoader, Dataset
class CustomData(Dataset):
  def __init__(self,features,targets):
    # Permute features from (N, H, W, C) to (N, C, H, W) for PyTorch Conv2d
    self.features = torch.tensor(features,dtype=torch.float32).permute(0, 3, 1, 2)
    self.targets = torch.tensor(targets,dtype=torch.long)
  def __getitem__(self,key):
    return self.features[key],self.targets[key]

  def __len__(self):
    return len(self.features)

In [22]:
import torch.nn as nn
train_dataset = CustomData(X_train,y_train)

# main class
class Model(nn.Module):
  def __init__(self):
        super().__init__()
        self.convLayer = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding='same'), # Corrected input channels from 32 to 3 for RGB images
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2,stride=2),

            nn.Conv2d(32,64,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        self.FClayer = nn.Sequential(
            nn.Flatten(),

            # Corrected input size for the first Linear layer based on output of conv layers (6x6x64)
            nn.Linear(8*8*64,128),
            nn.ReLU(),
            nn.Dropout(p=0.5),

            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(64,10)

        )

  def forward(self,x):
    x = self.convLayer(x)
    x = self.FClayer(x)
    return x

/tmp/ipykernel_2608/2904732064.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.features = torch.tensor(features,dtype=torch.float32).permute(0, 3, 1, 2)
/tmp/ipykernel_2608/2904732064.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.targets = torch.tensor(targets,dtype=torch.long)


In [23]:
from torch.optim import optimizer
from torch.utils import data
model = Model()
model = model.to(device)
learning_rate = 0.001
epochs = 40
optimizer = torch.optim.Adam(params=model.parameters(),lr=learning_rate,weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# mini batches
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)

# training loop
for epoch in range(epochs):
    for batch_features, batch_targets in train_loader:

        batch_features, batch_targets = batch_features.to(device), batch_targets.to(device)

        # forward pass
        X_train_tensor = batch_features
        y_train_tensor = batch_targets
        outputs = model(X_train_tensor)

        # loss calculation
        loss = criterion(outputs,y_train_tensor)

        # optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    print(f"Epoch: {epoch+1}, loss: {loss.item():.4f}")

Epoch: 1, loss: 1.4034
Epoch: 2, loss: 1.0469
Epoch: 3, loss: 1.8243
Epoch: 4, loss: 0.9870
Epoch: 5, loss: 0.7313
Epoch: 6, loss: 0.9310
Epoch: 7, loss: 0.5094
Epoch: 8, loss: 1.3774
Epoch: 9, loss: 0.6095
Epoch: 10, loss: 0.7370
Epoch: 11, loss: 1.0537
Epoch: 12, loss: 0.9642
Epoch: 13, loss: 0.3476
Epoch: 14, loss: 1.0285
Epoch: 15, loss: 0.3416
Epoch: 16, loss: 0.5933
Epoch: 17, loss: 0.4849
Epoch: 18, loss: 0.5566
Epoch: 19, loss: 0.3106
Epoch: 20, loss: 0.4098
Epoch: 21, loss: 0.3176
Epoch: 22, loss: 0.4109
Epoch: 23, loss: 0.6398
Epoch: 24, loss: 0.2556
Epoch: 25, loss: 0.5399
Epoch: 26, loss: 0.6935
Epoch: 27, loss: 0.2996
Epoch: 28, loss: 0.3632
Epoch: 29, loss: 0.7302
Epoch: 30, loss: 0.6295
Epoch: 31, loss: 0.3483
Epoch: 32, loss: 0.0969
Epoch: 33, loss: 0.3839
Epoch: 34, loss: 0.2513
Epoch: 35, loss: 0.5545
Epoch: 36, loss: 0.4293
Epoch: 37, loss: 0.3811
Epoch: 38, loss: 0.2291
Epoch: 39, loss: 0.4250
Epoch: 40, loss: 0.2563


In [24]:
test_dataset = CustomData(X_test,y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model.eval() # Set the model to evaluation mode
correct_predictions = 0
total_samples = 0

with torch.no_grad(): # Disable gradient calculation for evaluation
    for batch_features, batch_targets in test_loader:
        batch_features, batch_targets = batch_features.to(device), batch_targets.to(device)

        outputs = model(batch_features)
        _, predicted = torch.max(outputs.data, 1)

        total_samples += batch_targets.size(0)
        correct_predictions += (predicted == batch_targets).sum().item()

accuracy = (correct_predictions / total_samples) * 100
print(f'Accuracy of the model on the test images: {accuracy:.2f}%')

/tmp/ipykernel_2608/2904732064.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.features = torch.tensor(features,dtype=torch.float32).permute(0, 3, 1, 2)
/tmp/ipykernel_2608/2904732064.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.targets = torch.tensor(targets,dtype=torch.long)


Accuracy of the model on the test images: 74.70%


In [25]:
model.eval() # Set the model to evaluation mode
correct_predictions_train = 0
total_samples_train = 0

with torch.no_grad(): # Disable gradient calculation for evaluation
    for batch_features_train, batch_targets_train in train_loader:
        batch_features_train, batch_targets_train = batch_features_train.to(device), batch_targets_train.to(device)

        outputs_train = model(batch_features_train)
        _, predicted_train = torch.max(outputs_train.data, 1)

        total_samples_train += batch_targets_train.size(0)
        correct_predictions_train += (predicted_train == batch_targets_train).sum().item()

accuracy_train = (correct_predictions_train / total_samples_train) * 100
print(f'Accuracy of the model on the training images: {accuracy_train:.2f}%')

Accuracy of the model on the training images: 95.97%
